# Дообучение RuBert для определения спам сообщений

## Импорт необходимых библиотек

In [1]:
import numpy as np
import pandas as pd
import torch
from copy import deepcopy
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import torch.nn.functional as F

In [3]:
import sys
sys.path.append('..')
from utils.text import preprocess_text

## Чтение обработанного датасета

In [4]:
df = pd.read_csv('../data/preprocessed.csv', index_col=0)

In [5]:
df.info()

<class 'pandas.DataFrame'>
Index: 58067 entries, 0 to 58196
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   text               58067 non-null  str    
 1   label              58067 non-null  int64  
 2   emojis             58067 non-null  int64  
 3   newlines           58067 non-null  int64  
 4   whitespaces        58067 non-null  int64  
 5   links              58067 non-null  int64  
 6   tags               58067 non-null  int64  
 7   length             58067 non-null  int64  
 8   capital_ratio      58067 non-null  float64
 9   text_preprocessed  58067 non-null  str    
dtypes: float64(1), int64(7), str(2)
memory usage: 35.8 MB


In [6]:
df.head()

,text,label,emojis,newlines,whitespaces,links,tags,length,capital_ratio,text_preprocessed
0,Добрый день! Отличается ли перечень необходимы...,0,0,0,0,0,0,83,0.024096,добрый день отличается ли перечень необходимых...
1,Узбекистан. Рассматриваются обе формы,0,0,0,0,0,0,37,0.054054,узбекистан рассматриваются обе формы
2,"Здравствуйте, а как проходит поступление после...",0,0,0,0,0,0,147,0.040816,здравствуйте а как проходит поступление после ...
3,"Здравствуйте, а когда будет день открытых двер...",0,0,0,0,0,0,62,0.016129,здравствуйте а когда будет день открытых двере...
4,"Здравствуйте, необходимо для посещения дня отк...",0,0,0,0,0,0,82,0.012195,здравствуйте необходимо для посещения дня откр...


### Датасет без обработки

In [7]:
only_text = deepcopy(df[['text', 'label', 'text_preprocessed']])

In [8]:
dataset = only_text.to_dict('records')

In [9]:
def is_mostly_cyrillic(text, threshold=0.3):
    """Проверяет, содержит ли текст достаточную долю кириллических символов."""
    if not text:
        return False
    alpha_chars = [c for c in text if c.isalpha()]
    if not alpha_chars:
        return False
    cyrillic = sum(1 for c in alpha_chars if '\u0400' <= c <= '\u04FF')
    return (cyrillic / len(alpha_chars)) >= threshold

In [10]:
ru_dataset = []

In [11]:
for d in dataset:
    text = d.get('text', '') or ''
    text_p = d.get('text_preprocessed', '') or ''
    if is_mostly_cyrillic(text) or is_mostly_cyrillic(text_p):
        ru_dataset.append({"text": d['text'], "label": d['label']})

In [12]:
len(ru_dataset)

51364

In [13]:
ru_dataset

[{'text': 'Добрый день! Отличается ли перечень необходимых документов для иностранных граждан?',
  'label': 0},
 {'text': 'Узбекистан. Рассматриваются обе формы', 'label': 0},
 {'text': 'Здравствуйте, а как проходит поступление после колледжа? Внутренний экзамен или только ЕГЭ? И возможно ли всё ещё поступление сразу на второй курс?',
  'label': 0},
 {'text': 'Здравствуйте, а когда будет день открытых дверей и во сколько?',
  'label': 0},
 {'text': 'Здравствуйте, необходимо для посещения дня открытым дверей где-то регистрироваться',
  'label': 0},
 {'text': 'Добрый день! Да, необходимо пройти быструю регистрацию на сайте https://info.stankin.ru/dod',
  'label': 0},
 {'text': 'Лучшая специальность! Жаль с моими баллами не пройти(', 'label': 0},
 {'text': 'Было бы неплохо, если бы прикладывались ссылки для просмотра результатов экзаменов.)',
  'label': 0},
 {'text': 'Здравствуйте. Не смог найти списки поступающих на сайте. Ещё не опубликованы? Спасибо',
  'label': 0},
 {'text': 'теперь у

### Датасет с обработкой

In [14]:
only_text_p = deepcopy(df[['text_preprocessed', 'label']])

In [15]:
dataset_p = only_text_p.to_dict('records')

In [16]:
ru_dataset_p = []

In [17]:
ru_dataset_p = []
for d in dataset_p:
    text_p = d.get('text_preprocessed', '') or ''
    if is_mostly_cyrillic(text_p):
        ru_dataset_p.append({"text": d['text_preprocessed'], "label": d['label']})

In [18]:
len(dataset), len(ru_dataset), len(ru_dataset_p)

(58067, 51364, 51364)

## Дообучение RuBert

### Загрузка модели и токенайзера

In [19]:
MODEL_NAME = "cointegrated/rubert-tiny2"
# MODEL_NAME = "DeepPavlov/rubert-base-cased"

In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
rb_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)  # num_labels=2 для бинарной классификации

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai

### Подготовка выборок

In [21]:
def tokenize_function(examples):  # Почти лямбда функция для токенизации данных
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

#### Без предобработки

Загрузка датасета без предобработки

In [22]:
ru_df = pd.json_normalize(ru_dataset)

Преобразование в формат Hugging Face

In [23]:
dataset_hf = Dataset.from_pandas(ru_df)

Разделение на обучающую и тестовую выборки

In [24]:
rb_train_test_split = dataset_hf.train_test_split(test_size=0.2, seed=42)
rb_train_dataset = rb_train_test_split['train']
rb_test_dataset = rb_train_test_split['test']

In [25]:
ru_df_spam = ru_df.loc[ru_df['label'] == 1]

In [26]:
ru_df_spam

,text,label
22021,"⚡️ ОФИЦИАЛЬНОЕ ТЕЛЕГРАМ КАЗИНО, ИГРАЙ ПРЯМО В ...",1
22022,"🐶 ТОП 4 КАЗИНО, С САМЫМИ ВЫСОКИМИ ШАНСАМИ ВЫИГ...",1
22023,Только что выиграл 26000 рублей в новом телегр...,1
22024,"🍬 4 ОФИЦИАЛЬНЫХ КАЗИНО В ТЕЛЕГРАММЕ, С САМЫМИ ...",1
22025,ЗАПУСТИЛИ ОФИЦИАЛЬНОЕ КАЗИНО С ОГРОМНЫМ ШАНСОМ...,1
...,...,...
51359,"Лень закаляет характер, если вспомнить, скольк...",1
51360,Я гриб. Подскажите мне полагается общежитие?,1
51361,"Надо уметь закрывать скучную книгу, уходить с ...",1
51362,"⚡️⚡️⚡️\n\nДорогие родители, напоминаем вам, чт...",1


Токенизация данных

In [27]:
rb_tokenized_train = rb_train_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/41091 [00:00<?, ? examples/s]

In [28]:
rb_tokenized_test = rb_test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/10273 [00:00<?, ? examples/s]

#### С предобработкой

Загрузка датасета c предобработкой

In [29]:
ru_df_p = pd.json_normalize(ru_dataset_p)

Преобразование в формат Hugging Face

In [30]:
dataset_hf_p = Dataset.from_pandas(ru_df_p)

Разделение на обучающую и тестовую выборки

In [31]:
rb_train_test_split_p = dataset_hf_p.train_test_split(test_size=0.2, seed=42)
rb_train_dataset_p = rb_train_test_split_p['train']
rb_test_dataset_p = rb_train_test_split_p['test']

In [32]:
ru_df_spam_p = ru_df_p.loc[ru_df_p['label'] == 1]

In [33]:
ru_df_spam_p

,text,label
22021,официальное телеграм казино играй прямо в боте,1
22022,топ 4 казино с самыми высокими шансами выигрыш...,1
22023,только что выиграл 26000 рублей в новом телегр...,1
22024,4 официальных казино в телеграмме с самыми выс...,1
22025,запустили официальное казино с огромным шансом...,1
...,...,...
51359,лень закаляет характер если вспомнить сколько ...,1
51360,я гриб подскажите мне полагается общежитие,1
51361,надо уметь закрывать скучную книгу уходить с п...,1
51362,дорогие родители напоминаем вам что ровно чере...,1


Токенизация данных

In [34]:
rb_tokenized_train_p = rb_train_dataset_p.map(tokenize_function, batched=True)

Map:   0%|          | 0/41091 [00:00<?, ? examples/s]

In [35]:
rb_tokenized_test_p = rb_test_dataset_p.map(tokenize_function, batched=True)

Map:   0%|          | 0/10273 [00:00<?, ? examples/s]

### Подготовка trainer'а

In [36]:
train_loader = DataLoader(rb_tokenized_train, batch_size=16, shuffle=True)
test_loader = DataLoader(rb_tokenized_test, batch_size=16)

In [37]:
train_loader_p = DataLoader(rb_tokenized_train_p, batch_size=16, shuffle=True)
test_loader_p = DataLoader(rb_tokenized_test_p, batch_size=16)

In [38]:
def compute_metrics(eval_pred):
    """Вычисление метрик качества на каждой эпохе."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='binary'),
        'precision': precision_score(labels, predictions, average='binary'),
        'recall': recall_score(labels, predictions, average='binary'),
    }

In [39]:
class FocalLossTrainer(Trainer):
    """Trainer с Focal Loss для лучшей работы с hard examples."""
    def __init__(self, focal_alpha=0.25, focal_gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        ce_loss = F.cross_entropy(logits, labels, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.focal_alpha * (1 - pt) ** self.focal_gamma * ce_loss
        loss = focal_loss.mean()
        return (loss, outputs) if return_outputs else loss

In [40]:
rb_training_args = TrainingArguments(
    output_dir="rb_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=7,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_dir="rb_logs",
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [41]:
rb_trainer = FocalLossTrainer(
    focal_alpha=0.25,
    focal_gamma=2.0,
    model=rb_model,
    args=rb_training_args,
    train_dataset=rb_tokenized_train,
    eval_dataset=rb_tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [42]:
rb_trainer_p = FocalLossTrainer(
    focal_alpha=0.25,
    focal_gamma=2.0,
    model=rb_model,
    args=rb_training_args,
    train_dataset=rb_tokenized_train_p,
    eval_dataset=rb_tokenized_test_p,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

### Запуск дообучения

#### Перемещение модели на GPU

In [43]:
# Проверка доступности GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

Используемое устройство: cuda


In [44]:
rb_model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-12, 

#### Тренировка

##### Без предобработки данных

In [45]:
rb_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.004907,0.002145,0.991921,0.988794,0.991606,0.985999
2,0.000775,0.001831,0.992310,0.989308,0.994558,0.984114
3,0.003254,0.001824,0.992797,0.990030,0.990831,0.989230
4,0.000203,0.002102,0.992894,0.990174,0.990040,0.990307
5,0.000209,0.001967,0.993575,0.991110,0.991644,0.990576
6,0.000098,0.002101,0.993965,0.991651,0.991918,0.991384
7,0.000051,0.002115,0.993867,0.991517,0.991651,0.991384


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.

TrainOutput(global_step=17983, training_loss=0.0018171988575985923, metrics={'train_runtime': 1229.0534, 'train_samples_per_second': 234.031, 'train_steps_per_second': 14.632, 'total_flos': 2121094771955712.0, 'train_loss': 0.0018171988575985923, 'epoch': 7.0})

In [48]:
from transformers.utils.notebook import NotebookProgressCallback

In [49]:
rb_trainer.remove_callback(NotebookProgressCallback)
rb_trainer.evaluate()

{'eval_loss': 0.00210074195638299,
 'eval_accuracy': 0.9939647619974691,
 'eval_f1': 0.9916509561001885,
 'eval_precision': 0.9919181034482759,
 'eval_recall': 0.9913839526117394,
 'eval_runtime': 17.9835,
 'eval_samples_per_second': 571.246,
 'eval_steps_per_second': 35.755,
 'epoch': 7.0}

Сохранение модели

In [50]:
rb_model.save_pretrained("../models/finetuned_rubert_tiny2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [51]:
tokenizer.save_pretrained("../models/finetuned_rubert_tiny2")

('../models/finetuned_rubert_tiny2\\tokenizer_config.json',
 '../models/finetuned_rubert_tiny2\\tokenizer.json')

##### С предобработкой данных

In [52]:
rb_trainer_p.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.005578,0.001641,0.993575,0.991129,0.989533,0.992730
2,0.000037,0.001648,0.993770,0.991363,0.993777,0.988961
3,0.006939,0.001965,0.994062,0.991796,0.990863,0.992730
4,0.000048,0.002271,0.993283,0.990725,0.989262,0.992192
5,0.000020,0.002325,0.994062,0.991785,0.992185,0.991384


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.

TrainOutput(global_step=12845, training_loss=0.000550787222307315, metrics={'train_runtime': 819.7865, 'train_samples_per_second': 350.868, 'train_steps_per_second': 21.936, 'total_flos': 1515067694254080.0, 'train_loss': 0.000550787222307315, 'epoch': 5.0})

In [53]:
rb_trainer_p.remove_callback(NotebookProgressCallback)
rb_trainer_p.evaluate()

{'eval_loss': 0.0019651020411401987,
 'eval_accuracy': 0.994062104545897,
 'eval_f1': 0.9917955615332885,
 'eval_precision': 0.9908626713249127,
 'eval_recall': 0.9927302100161551,
 'eval_runtime': 17.44,
 'eval_samples_per_second': 589.048,
 'eval_steps_per_second': 36.869,
 'epoch': 5.0}

Сохранение модели

In [54]:
rb_model.save_pretrained("../models/finetuned_rubert_tiny2_p")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [55]:
tokenizer.save_pretrained("../models/finetuned_rubert_tiny2_p")

('../models/finetuned_rubert_tiny2_p\\tokenizer_config.json',
 '../models/finetuned_rubert_tiny2_p\\tokenizer.json')

## Оценка качества моделей

Тестовые сообщения

In [56]:
test_messages = [
    "Это честное сообщение от пользователя.",
    "🔥 Казино онлайн! Зарабатывай миллионы прямо сейчас! 💰💎",
    "Зарабатывай миллионы **онлайн** прямо сейчас!",
    "Работа на дому, легкий доход. Пиши в личку!",
    "Привет! Как дела? У меня всё отлично.",
    "Discover the hidden secrets of the digital market that top traders don’t want you to know! I’m seeking five motivated individuals who are committed to earning over $100K weekly in the digital market. Once you start seeing profits, I’ll require just 15% of your earnings as my fee. Please note: I’m only interested in working with five serious and dedicated people should send me a direct message or ask me (HOW) via TELEGRAM\n\nhttps://t.me/ancleroyofficial",
    "Discover the hidden secrets of the digital market that top traders don’t want you to know! I’m seeking five motivated individuals who are committed to earning over $100K weekly in the digital market. Once you start seeing profits, I’ll require just 15% of your earnings as my fee. Please note: I’m only interested in working with five serious and dedicated people should send me a direct message or click the link on my bio",
    "steam gift 50$ - steamcommunity.com/gift-card/pay/50\n@everyone",
    "Давайте **вместе** будем писать про казино в чатах!!! Присоединяйтесь!",
    "Как же надоели эти сообщения про казино",
    "Добрый день. Для подачи документов необходимо пройти регистрацию здесь: stankin.ru",
    "Добрый день. Для подачи документов необходимо пройти регистрацию здесь: https://stankin.ru",
    "Поступление – это почти что казино! Лотерея!",
    "3-4 часа и 8 тысяч твои!  Пиши  https://t.me/rasmuswork1",
    "Выиграл 345к дало x3450\n\nИграл тут: @jet_casino_ibot"
]

In [57]:
classifier = pipeline("text-classification", model="../models/finetuned_rubert_tiny2", tokenizer="../models/finetuned_rubert_tiny2")

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

In [58]:
classifier_p = pipeline("text-classification", model="../models/finetuned_rubert_tiny2_p", tokenizer="../models/finetuned_rubert_tiny2_p")

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

In [59]:
for message in test_messages:
    result = classifier(message)
    print(message)
    print(result)
    print()

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Это честное сообщение от пользователя.
[{'label': 'LABEL_0', 'score': 0.887690544128418}]

🔥 Казино онлайн! Зарабатывай миллионы прямо сейчас! 💰💎
[{'label': 'LABEL_1', 'score': 0.9791872501373291}]

Зарабатывай миллионы **онлайн** прямо сейчас!
[{'label': 'LABEL_1', 'score': 0.9497878551483154}]

Работа на дому, легкий доход. Пиши в личку!
[{'label': 'LABEL_1', 'score': 0.9905822277069092}]

Привет! Как дела? У меня всё отлично.
[{'label': 'LABEL_0', 'score': 0.9145657420158386}]

Discover the hidden secrets of the digital market that top traders don’t want you to know! I’m seeking five motivated individuals who are committed to earning over $100K weekly in the digital market. Once you start seeing profits, I’ll require just 15% of your earnings as my fee. Please note: I’m only interested in working with five serious and dedicated people should send me a direct message or ask me (HOW) via TELEGRAM

https://t.me/ancleroyofficial
[{'label': 'LABEL_1', 'score': 0.9704909920692444}]

Disco

In [60]:
for message in test_messages:
    result = classifier_p(message)
    print(message)
    print(result)
    print()

Это честное сообщение от пользователя.
[{'label': 'LABEL_1', 'score': 0.705121636390686}]

🔥 Казино онлайн! Зарабатывай миллионы прямо сейчас! 💰💎
[{'label': 'LABEL_1', 'score': 0.9829283356666565}]

Зарабатывай миллионы **онлайн** прямо сейчас!
[{'label': 'LABEL_1', 'score': 0.9838969707489014}]

Работа на дому, легкий доход. Пиши в личку!
[{'label': 'LABEL_1', 'score': 0.9892311692237854}]

Привет! Как дела? У меня всё отлично.
[{'label': 'LABEL_0', 'score': 0.5328430533409119}]

Discover the hidden secrets of the digital market that top traders don’t want you to know! I’m seeking five motivated individuals who are committed to earning over $100K weekly in the digital market. Once you start seeing profits, I’ll require just 15% of your earnings as my fee. Please note: I’m only interested in working with five serious and dedicated people should send me a direct message or ask me (HOW) via TELEGRAM

https://t.me/ancleroyofficial
[{'label': 'LABEL_1', 'score': 0.9851539134979248}]

Disco